# DaTSCAN — Fase 4: CNN 3D local de dos canales

CNN tridimensional sobre un recorte estriatal `48×48×32`. El canal 1 contiene intensidad y el canal 2 la diferencia absoluta izquierda-derecha. El fold externo nunca se usa para *early stopping*: cada entrenamiento crea una validación interna estratificada dentro de sus datos de entrenamiento.

Por defecto se ejecutan primero los cinco folds agrupados. La CV estratificada queda disponible como experimento posterior.

## 0. Dependencias

In [ ]:
# Descomente si PyTorch no está instalado. Para GPU instale la versión compatible con su CUDA.
# %pip install torch numpy pandas scikit-learn matplotlib seaborn

## 1. Librerías, reproducibilidad y configuración

In [ ]:
from pathlib import Path
import os
import copy, gc, random, time, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore',category=FutureWarning)
sns.set_theme(style='whitegrid')
SEED=20260910
def seed_everything(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_everything()
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__,'| dispositivo:',DEVICE)
if DEVICE.type=='cuda':print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT=Path(r'C:\Users\DELL\OneDrive\Escritorio\kaggle\parkinson')
PROJECT_DIR=DATA_ROOT/'latent_protocol_cv'
PREPROCESS_DIR=PROJECT_DIR/'preprocessed_96x96x64_v2'
FOLDS_CSV=PROJECT_DIR/'outputs'/'train_protocol_folds.csv'
ROI_CSV=PROJECT_DIR/'roi_adaptive_v3'/'roi_features.csv'
# Carpeta nueva para no reutilizar el fold generado con la configuración anterior.
CNN_DIR=PROJECT_DIR/'cnn3d_local_2ch_v1'
CNN_DIR.mkdir(parents=True,exist_ok=True)

LOCAL_SHAPE=(48,48,32)
BATCH_SIZE=4  # el smoke test de sobreajuste confirmó que cabe en memoria
NUM_WORKERS=0  # seguro para Windows/Positron
MAX_EPOCHS=40
PATIENCE=7
LEARNING_RATE=1e-3
WEIGHT_DECAY=1e-5
INNER_VALID_FRACTION=0.12
RUN_GROUPED=True
RUN_STRATIFIED=False  # activar solo después de terminar y revisar la CV agrupada

print('Batch:',BATCH_SIZE,'| salida:',CNN_DIR)
for required_path in (PREPROCESS_DIR,FOLDS_CSV,ROI_CSV):
    print(required_path,'| existe:',required_path.exists())
    if not required_path.exists():raise FileNotFoundError(required_path)

## 2. Manifiesto y controles

In [ ]:
manifest=pd.read_csv(FOLDS_CSV)
roi_locations=pd.read_csv(ROI_CSV)
location_columns=['uid','midline_x','left_y','right_y','z_peak']
manifest=manifest.merge(roi_locations[location_columns],on='uid',how='left',validate='one_to_one')
manifest['processed_path']=manifest['uid'].map(lambda uid:str((PREPROCESS_DIR/f'{uid}.npz').resolve()))
missing=manifest.loc[~manifest.processed_path.map(lambda p:Path(p).exists())]
if len(missing):raise FileNotFoundError(f'Faltan {len(missing)} volúmenes.')
if manifest[location_columns[1:]].isna().any().any():raise ValueError('Faltan coordenadas de localización ROI.')
if manifest.uid.duplicated().any():raise ValueError('UID duplicados.')
if (manifest.groupby('protocol_cluster').fold.nunique()>1).any():raise ValueError('Un clúster está dividido entre folds.')
print('Estudios:',len(manifest),'| normales:',int((manifest.target==0).sum()),'| patológicos:',int((manifest.target==1).sum()))
display(manifest.groupby('fold').agg(n=('uid','size'),prevalence=('target','mean'),clusters=('protocol_cluster',lambda x:sorted(x.unique().tolist()))))

## 3. Dataset y aumentos

In [ ]:
def crop_pad(volume,center,shape=LOCAL_SHAPE):
    output=np.zeros(shape,dtype=np.float32)
    center=np.rint(center).astype(int);starts=center-np.asarray(shape)//2
    src=[];dst=[]
    for start,size,source_size in zip(starts,shape,volume.shape):
        src_start,src_end=max(0,start),min(source_size,start+size)
        dst_start=max(0,-start);dst_end=dst_start+max(0,src_end-src_start)
        src.append(slice(src_start,src_end));dst.append(slice(dst_start,dst_end))
    output[tuple(dst)]=volume[tuple(src)]
    return output

class DaTSCANDataset(Dataset):
    def __init__(self,frame,augment=False):
        self.frame=frame.reset_index(drop=True);self.augment=augment
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with np.load(row.processed_path) as saved:volume=saved['volume'].astype(np.float32)
        center=(float(row.midline_x),float((row.left_y+row.right_y)/2),float(row.z_peak))
        local=crop_pad(volume,center)
        asymmetry=np.abs(local-local[::-1,:,:])
        x=torch.from_numpy(np.stack([local,asymmetry],axis=0))
        if self.augment:
            # Reflexión izquierda-derecha: conserva la etiqueta y evita preferencia lateral.
            if torch.rand(())<0.5:x=torch.flip(x,dims=(1,))
            # Baseline: perturbación de intensidad mínima; se ampliará solo si aprende.
            scale=float(torch.empty(1).uniform_(0.97,1.03))
            x=x*scale
            x=torch.clamp(x,0,1)
        return x,torch.tensor(float(row.target),dtype=torch.float32),str(row.uid)

def make_loader(frame,augment,shuffle,batch_size=BATCH_SIZE):
    return DataLoader(DaTSCANDataset(frame,augment),batch_size=batch_size,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False)

## 4. Arquitectura compacta

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        # GroupNorm exige que out_channels sea divisible por groups.
        groups=max(g for g in (8,4,2,1) if out_channels%g==0)
        self.block=nn.Sequential(
            nn.Conv3d(in_channels,out_channels,3,stride=2,padding=1,bias=False),
            nn.GroupNorm(groups,out_channels),nn.SiLU(inplace=True),
            nn.Conv3d(out_channels,out_channels,3,padding=1,bias=False),
            nn.GroupNorm(groups,out_channels),nn.SiLU(inplace=True))
    def forward(self,x):return self.block(x)

class SmallCNN3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(ConvBlock(2,16),ConvBlock(16,32),ConvBlock(32,64))
        self.head=nn.Sequential(nn.Linear(128,64),nn.SiLU(),nn.Dropout(0.25),nn.Linear(64,1))
    def forward(self,x):
        x=self.features(x)
        pooled=torch.cat([F.adaptive_avg_pool3d(x,1),F.adaptive_max_pool3d(x,1)],dim=1).flatten(1)
        return self.head(pooled).squeeze(1)

model=SmallCNN3D().to(DEVICE)
print('Parámetros:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Smoke test obligatorio

In [ ]:
smoke_frame=manifest.sample(n=min(8,len(manifest)),random_state=SEED)
smoke_loader=make_loader(smoke_frame,augment=True,shuffle=False,batch_size=min(BATCH_SIZE,4))
x_smoke,y_smoke,_=next(iter(smoke_loader))
smoke_model=SmallCNN3D().to(DEVICE)
optimizer=torch.optim.AdamW(smoke_model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
optimizer.zero_grad(set_to_none=True)
logits=smoke_model(x_smoke.to(DEVICE));loss=F.binary_cross_entropy_with_logits(logits,y_smoke.to(DEVICE))
loss.backward();optimizer.step()
print('Entrada:',tuple(x_smoke.shape),'| salida:',tuple(logits.shape),'| loss:',float(loss.detach().cpu()))
fig,axes=plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(x_smoke[0,0,:,:,LOCAL_SHAPE[2]//2].T,cmap='inferno',origin='lower',vmin=0,vmax=1)
axes[0].set_title('Canal 1: intensidad local')
axes[1].imshow(x_smoke[0,1,:,:,LOCAL_SHAPE[2]//2].T,cmap='magma',origin='lower',vmin=0,vmax=1)
axes[1].set_title('Canal 2: diferencia I-D')
for ax in axes:ax.axis('off')
plt.tight_layout();plt.show()
del smoke_model,optimizer,x_smoke,y_smoke,logits,loss;gc.collect()
if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 6. Funciones de entrenamiento y predicción

In [ ]:
@torch.no_grad()
def predict_loader(model,loader):
    model.eval();predictions=[];targets=[];uids=[]
    for x,y,batch_uids in loader:
        logits=model(x.to(DEVICE,non_blocking=True))
        predictions.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        targets.extend(y.numpy().tolist());uids.extend(list(batch_uids))
    return np.asarray(predictions),np.asarray(targets,dtype=int),uids

def run_outer_fold(train_frame,outer_valid_frame,scheme,fold):
    seed_everything(SEED+fold)
    splitter=StratifiedShuffleSplit(n_splits=1,test_size=INNER_VALID_FRACTION,random_state=SEED+fold)
    inner_train_idx,inner_valid_idx=next(splitter.split(train_frame,train_frame.target))
    inner_train=train_frame.iloc[inner_train_idx];inner_valid=train_frame.iloc[inner_valid_idx]
    train_loader=make_loader(inner_train,True,True)
    inner_loader=make_loader(inner_valid,False,False)
    outer_loader=make_loader(outer_valid_frame,False,False)
    model=SmallCNN3D().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=2,min_lr=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))
    best_loss=np.inf;best_state=None;best_epoch=0;wait=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();running_loss=0.0;n_seen=0
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                logits=model(x);batch_loss=F.binary_cross_entropy_with_logits(logits,y)
            scaler.scale(batch_loss).backward()
            scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer);scaler.update()
            running_loss+=float(batch_loss.detach().cpu())*len(y);n_seen+=len(y)
        inner_pred,inner_y,_=predict_loader(model,inner_loader)
        inner_pred=np.clip(inner_pred,1e-6,1-1e-6);inner_loss=log_loss(inner_y,inner_pred)
        scheduler.step(inner_loss)
        history.append({'scheme':scheme,'fold':fold,'epoch':epoch,'train_loss':running_loss/n_seen,
                        'inner_logloss':inner_loss,'inner_auc':roc_auc_score(inner_y,inner_pred),
                        'learning_rate':optimizer.param_groups[0]['lr']})
        print(f'{scheme} F{fold} E{epoch:02d} | train={running_loss/n_seen:.4f} | inner={inner_loss:.4f}')
        if inner_loss<best_loss-1e-4:
            best_loss=inner_loss;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict());wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    outer_pred,outer_y,outer_uids=predict_loader(model,outer_loader)
    outer_pred=np.clip(outer_pred,1e-6,1-1e-6)
    metrics={'scheme':scheme,'fold':fold,'n_train':len(train_frame),'n_valid':len(outer_valid_frame),
             'best_epoch':best_epoch,'inner_best_logloss':best_loss,
             'logloss':log_loss(outer_y,outer_pred),'auc':roc_auc_score(outer_y,outer_pred),
             'brier':brier_score_loss(outer_y,outer_pred),'valid_prevalence':float(outer_y.mean())}
    return model,pd.DataFrame(history),pd.DataFrame({'uid':outer_uids,'target':outer_y,'prediction':outer_pred}),metrics


## 7. Ejecutar una estrategia de CV
Cada fold guarda inmediatamente pesos, historial y predicciones. Si esas predicciones ya existen, el fold se reutiliza.

In [ ]:
def run_cv(scheme,splits):
    oof=np.full(len(manifest),np.nan);metric_rows=[]
    uid_to_index={uid:i for i,uid in enumerate(manifest.uid)}
    for fold,(train_idx,valid_idx) in enumerate(splits):
        pred_path=CNN_DIR/f'{scheme}_fold{fold}_predictions.csv'
        if pred_path.exists():
            fold_pred=pd.read_csv(pred_path);reused=True
            valid_uids=set(manifest.iloc[valid_idx].uid)
            if set(fold_pred.uid)!=valid_uids:raise ValueError(f'Predicciones incompatibles en {pred_path}')
            y_fold=fold_pred.target.astype(int).to_numpy();p_fold=fold_pred.prediction.to_numpy()
            metrics={'scheme':scheme,'fold':fold,'n_train':len(train_idx),'n_valid':len(valid_idx),'best_epoch':np.nan,
                     'inner_best_logloss':np.nan,'logloss':log_loss(y_fold,p_fold),'auc':roc_auc_score(y_fold,p_fold),
                     'brier':brier_score_loss(y_fold,p_fold),'valid_prevalence':float(y_fold.mean()),'reused':True}
        else:
            model,history,fold_pred,metrics=run_outer_fold(manifest.iloc[train_idx],manifest.iloc[valid_idx],scheme,fold)
            torch.save(model.state_dict(),CNN_DIR/f'{scheme}_fold{fold}.pt')
            history.to_csv(CNN_DIR/f'{scheme}_fold{fold}_history.csv',index=False)
            fold_pred.to_csv(pred_path,index=False);metrics['reused']=False
            del model;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        for uid,pred in zip(fold_pred.uid,fold_pred.prediction):oof[uid_to_index[uid]]=pred
        metric_rows.append(metrics);pd.DataFrame(metric_rows).to_csv(CNN_DIR/f'{scheme}_fold_metrics_checkpoint.csv',index=False)
        display(pd.DataFrame([metrics]))
    if np.isnan(oof).any():raise RuntimeError('OOF incompleto.')
    overall={'scheme':scheme,'n':len(oof),'logloss_oof':log_loss(manifest.target,oof),
             'auc_oof':roc_auc_score(manifest.target,oof),'brier_oof':brier_score_loss(manifest.target,oof)}
    pd.DataFrame({'uid':manifest.uid,'target':manifest.target,'protocol_cluster':manifest.protocol_cluster,
                  'fold':manifest.fold,'prediction':oof}).to_csv(CNN_DIR/f'{scheme}_oof.csv',index=False)
    pd.DataFrame(metric_rows).to_csv(CNN_DIR/f'{scheme}_fold_metrics.csv',index=False)
    return oof,pd.DataFrame(metric_rows),overall

## 8. CV agrupada por protocolo — experimento principal

In [ ]:
grouped_output=None
if RUN_GROUPED:
    grouped_splits=[]
    for fold in sorted(manifest.fold.unique()):
        valid_idx=np.flatnonzero(manifest.fold.to_numpy()==fold)
        train_idx=np.flatnonzero(manifest.fold.to_numpy()!=fold)
        grouped_splits.append((train_idx,valid_idx))
    grouped_output=run_cv('protocol_grouped',grouped_splits)
    display(pd.DataFrame([grouped_output[2]]))
    display(grouped_output[1])

## 9. CV estratificada — ejecutar después
Cambie `RUN_STRATIFIED=True` en configuración únicamente después de guardar y revisar la CV agrupada.

In [ ]:
stratified_output=None
if RUN_STRATIFIED:
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
    stratified_output=run_cv('stratified',list(skf.split(manifest,manifest.target)))
    display(pd.DataFrame([stratified_output[2]]))
    display(stratified_output[1])

## 10. Diagnóstico por protocolo

In [ ]:
if grouped_output is not None:
    grouped_oof=grouped_output[0]
    protocol_rows=[]
    for cluster,indices in manifest.groupby('protocol_cluster').groups.items():
        idx=np.asarray(list(indices));yt=manifest.target.to_numpy()[idx];pred=grouped_oof[idx]
        protocol_rows.append({'protocol_cluster':cluster,'n':len(idx),'prevalence':yt.mean(),
                              'logloss':log_loss(yt,pred),'auc':roc_auc_score(yt,pred),'brier':brier_score_loss(yt,pred)})
    protocol_metrics=pd.DataFrame(protocol_rows)
    protocol_metrics.to_csv(CNN_DIR/'protocol_grouped_metrics_by_cluster.csv',index=False)
    display(protocol_metrics)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    sns.barplot(data=protocol_metrics,x='protocol_cluster',y='logloss',ax=axes[0],color='#386641')
    sns.barplot(data=protocol_metrics,x='protocol_cluster',y='auc',ax=axes[1],color='#6A4C93')
    axes[1].axhline(.5,color='crimson',linestyle='--');plt.tight_layout();plt.show()